In [0]:
dbutils.widgets.text("password", "")


In [0]:
from pyspark.sql.functions import (
    col, lit, concat_ws, when, year, month, quarter,
    dayofmonth, dayofweek, date_format, datediff, current_date,
    explode, sequence, to_date, row_number
)
from pyspark.sql.types import DecimalType, DateType, IntegerType
from pyspark.sql.window import Window
from datetime import date

SILVER = "/Volumes/workspace/default/silver"
TODAY  = date.today()

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.dw")
print(f"Target: workspace.dw (Delta)  |  Run date: {TODAY}")


In [0]:
BD_HOLIDAYS = ["02-21","03-17","03-25","03-26","04-14","05-01","08-15","12-16","12-25"]

dim_date = (
    spark.range(1)
    .select(explode(sequence(to_date(lit("2020-01-01")), to_date(lit("2030-12-31")))).alias("FullDate"))
    .withColumn("DateSK",         (year("FullDate")*10000 + month("FullDate")*100 + dayofmonth("FullDate")).cast("int"))
    .withColumn("Year",           year("FullDate"))
    .withColumn("Month",          month("FullDate"))
    .withColumn("MonthName",      date_format("FullDate", "MMMM"))
    .withColumn("Quarter",        quarter("FullDate"))
    .withColumn("IsWeekend",      dayofweek("FullDate").isin([1, 7]).cast("boolean"))
    .withColumn("IsPublicHoliday",lit(False))
    .withColumn("FiscalYear",     when(month("FullDate") >= 7, year("FullDate")+1).otherwise(year("FullDate")))
)
for hd in BD_HOLIDAYS:
    dim_date = dim_date.withColumn("IsPublicHoliday",
        when(date_format("FullDate","MM-dd") == hd, lit(True)).otherwise(col("IsPublicHoliday"))
    )

dim_date = dim_date.select("DateSK","FullDate","Year","Month","MonthName","Quarter","IsWeekend","IsPublicHoliday","FiscalYear")
dim_date.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.dw.dim_date")
print(f"✓ dim_date: {dim_date.count()} rows")


In [0]:
dept_silver = spark.read.parquet(f"{SILVER}/department")

w = Window.orderBy("DepartmentId")
dim_dept = (
    dept_silver
    .select(col("department_id").alias("DepartmentId"), col("department_name").alias("DepartmentName"))
    .withColumn("DepartmentSK", row_number().over(w))
    .select("DepartmentSK","DepartmentId","DepartmentName")
)
dim_dept.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.dw.dim_department")
print(f"✓ dim_department: {dim_dept.count()} rows")


In [0]:
lt_silver = spark.read.parquet(f"{SILVER}/leavetype")

w = Window.orderBy("LeaveTypeId")
dim_lt = (
    lt_silver
    .select(
        col("leave_type_id").alias("LeaveTypeId"),
        col("leave_type_name").alias("LeaveTypeName"),
        col("default_days_per_year").alias("DefaultDaysPerYear"),
        col("is_carry_forward_allowed").alias("IsCarryForwardAllowed")
    )
    .withColumn("LeaveTypeSK", row_number().over(w))
    .select("LeaveTypeSK","LeaveTypeId","LeaveTypeName","DefaultDaysPerYear","IsCarryForwardAllowed")
)
dim_lt.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.dw.dim_leavetype")
print(f"✓ dim_leavetype: {dim_lt.count()} rows")


In [0]:
emp_silver  = spark.read.parquet(f"{SILVER}/employee")
dept_silver = spark.read.parquet(f"{SILVER}/department")

mgr_names  = emp_silver.select(col("employee_id").alias("m_id"), concat_ws(" ", col("first_name"), col("last_name")).alias("ManagerName"))
dept_names = dept_silver.select(col("department_id").alias("d_id"), col("department_name").alias("DepartmentName"))

incoming = (
    emp_silver
    .join(mgr_names,  emp_silver.manager_id   == col("m_id"), "left")
    .join(dept_names, emp_silver.department_id == col("d_id"), "left")
    .select(
        col("employee_id").alias("EmployeeId"),
        concat_ws(" ", col("first_name"), col("last_name")).alias("FullName"),
        col("DepartmentName"),
        col("ManagerName"),
        (datediff(current_date(), col("join_date")) / 365.25).cast(DecimalType(5,2)).alias("TenureYears"),
        col("join_date").alias("JoinDate")
    )
)

try:
    existing       = spark.read.table("workspace.dw.dim_employee")
    existing_count = existing.count()
except:
    existing_count = 0

print(f"Existing dim_employee rows: {existing_count}")

COLS = ["EmployeeId","FullName","DepartmentName","ManagerName","TenureYears","ValidFrom","ValidTo","IsCurrent"]

if existing_count == 0:
    dim_emp = (
        incoming
        .withColumn("ValidFrom", col("JoinDate"))
        .withColumn("ValidTo",   lit(None).cast(DateType()))
        .withColumn("IsCurrent", lit(True))
        .select(COLS)
    )
else:
    current_rows = existing.filter("IsCurrent = true").select(
        "EmployeeId", col("DepartmentName").alias("OldDept"), col("ValidFrom")
    )
    merged = incoming.join(current_rows, "EmployeeId", "left")

    unchanged = (
        merged.filter(col("OldDept").isNull() | (col("DepartmentName") == col("OldDept")))
        .withColumn("ValidFrom", when(col("ValidFrom").isNull(), col("JoinDate")).otherwise(col("ValidFrom")))
        .withColumn("ValidTo",   lit(None).cast(DateType()))
        .withColumn("IsCurrent", lit(True))
        .select(COLS)
    )

    changed_ids = [
        r.EmployeeId for r in
        merged.filter(col("OldDept").isNotNull() & (col("DepartmentName") != col("OldDept")))
        .select("EmployeeId").collect()
    ]

    expired = (
        existing.filter(col("IsCurrent") & col("EmployeeId").isin(changed_ids))
        .withColumn("ValidTo",   lit(TODAY).cast(DateType()))
        .withColumn("IsCurrent", lit(False))
        .select(COLS)
    )
    new_current = (
        merged.filter(col("EmployeeId").isin(changed_ids))
        .withColumn("ValidFrom", lit(TODAY).cast(DateType()))
        .withColumn("ValidTo",   lit(None).cast(DateType()))
        .withColumn("IsCurrent", lit(True))
        .select(COLS)
    )
    history = existing.filter("IsCurrent = false").select(COLS)

    dim_emp = unchanged.union(expired).union(new_current).union(history)

# Generate SK (ordered by EmployeeId + ValidFrom to keep history grouped)
w = Window.orderBy("EmployeeId","ValidFrom")
dim_emp = dim_emp.withColumn("EmployeeSK", row_number().over(w)).select("EmployeeSK",*COLS)

dim_emp.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.dw.dim_employee")
print(f"✓ dim_employee: {dim_emp.count()} rows  (current: {dim_emp.filter('IsCurrent = true').count()})")

# SK lookups for facts (reuse in-memory DataFrames — no need to re-read)
emp_sk_lookup = dim_emp.filter("IsCurrent = true").select(col("EmployeeId").alias("e_id"), col("EmployeeSK"))
lt_sk_lookup  = dim_lt.select(col("LeaveTypeId").alias("lt_id"), col("LeaveTypeSK"))


In [0]:
lr_silver = spark.read.parquet(f"{SILVER}/leaverequest")

fact_lr = (
    lr_silver
    .join(emp_sk_lookup, lr_silver.employee_id   == col("e_id"),  "left")
    .join(lt_sk_lookup,  lr_silver.leave_type_id == col("lt_id"), "left")
    .withColumn("StartDateSK",   (year("start_date")*10000 + month("start_date")*100 + dayofmonth("start_date")).cast("int"))
    .withColumn("EndDateSK",     (year("end_date")*10000   + month("end_date")*100   + dayofmonth("end_date")).cast("int"))
    .withColumn("DaysToApproval",
        when(col("approved_at").isNotNull(), datediff(col("approved_at"), col("created_at")))
        .otherwise(lit(None).cast("int"))
    )
    .select(
        col("leave_request_id").alias("LeaveRequestId"),
        col("EmployeeSK"), col("LeaveTypeSK"),
        col("StartDateSK"), col("EndDateSK"),
        col("total_days").alias("TotalDays"),
        col("status").alias("Status"),
        col("DaysToApproval")
    )
)
fact_lr.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.dw.fact_leaverequest")
print(f"✓ fact_leaverequest: {fact_lr.count()} rows")


In [0]:
lb_silver = spark.read.parquet(f"{SILVER}/leavebalance")

fact_lb = (
    lb_silver
    .join(emp_sk_lookup, lb_silver.employee_id   == col("e_id"),  "left")
    .join(lt_sk_lookup,  lb_silver.leave_type_id == col("lt_id"), "left")
    .withColumn("UtilisationRate",
        when(col("total_entitled") > 0, (col("total_used") / col("total_entitled")).cast(DecimalType(5,4)))
        .otherwise(lit(0).cast(DecimalType(5,4)))
    )
    .select(
        col("leave_balance_id").alias("LeaveBalanceId"),
        col("EmployeeSK"), col("LeaveTypeSK"),
        col("year").alias("Year"),
        col("total_entitled").alias("TotalEntitled"),
        col("total_used").alias("TotalUsed"),
        col("carry_forward").alias("CarryForward"),
        col("remaining").alias("Remaining"),
        col("UtilisationRate")
    )
)
fact_lb.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.dw.fact_leavebalance")
print(f"✓ fact_leavebalance: {fact_lb.count()} rows")


In [0]:
att_silver = spark.read.parquet(f"{SILVER}/attendance")

fact_att = (
    att_silver
    .join(emp_sk_lookup, att_silver.employee_id == col("e_id"), "left")
    .withColumn("DateSK",          (year("work_date")*10000 + month("work_date")*100 + dayofmonth("work_date")).cast("int"))
    .withColumn("IsLateArrival",
        when(col("is_present") & col("check_in_time").isNotNull(),  col("check_in_time")  > lit("09:15")).otherwise(lit(False))
    )
    .withColumn("IsEarlyDeparture",
        when(col("is_present") & col("check_out_time").isNotNull(), col("check_out_time") < lit("17:00")).otherwise(lit(False))
    )
    .select(
        col("EmployeeSK"), col("DateSK"),
        col("check_in_time").alias("CheckInTime"),
        col("check_out_time").alias("CheckOutTime"),
        col("is_present").alias("IsPresent"),
        col("IsLateArrival"), col("IsEarlyDeparture")
    )
)
fact_att.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.dw.fact_attendance")
print(f"✓ fact_attendance: {fact_att.count()} rows")


In [0]:
tables = [
    "workspace.dw.dim_date", "workspace.dw.dim_department",
    "workspace.dw.dim_leavetype", "workspace.dw.dim_employee",
    "workspace.dw.fact_leaverequest", "workspace.dw.fact_leavebalance",
    "workspace.dw.fact_attendance"
]
print("=== DWH Row Counts ===")
for t in tables:
    df = spark.read.table(t)
    print(f"{t}: {df.count()} rows")
